In [17]:
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt 
from matplotlib.backends.backend_pdf import PdfPages 
%matplotlib inline 

### 1. import data

In [18]:
train = pd.read_csv('train_data.csv')
test = pd.read_csv('test_data.csv')
val = pd.read_csv('val_data.csv')
c1 = ['Gender', 'Married', 'Age', 'CreditScore', 'Dependents', 'NumBankAccts', 'HasCrCard', 'EmergingMarketFund', 'RealEstate',
      'PrivateEquity', 'GovtBonds', 'CorpBonds', 'ETF Tech', 'ETF Health', 'ETF Med', 'Debt', 'Net Assets', 'Mortgage', 
      'EstimatedSalary', 'Portfolio Return', 'Diversification', 'BusinessOwner', 'Revenue', 'LifeInsurance', 
      'NumTransactions', 'DaysSinceLastTransaction', 'ForeignAssets', 'NumProducts']
c2 = [0, 0, -1, -1, -1, -1, -1, -1, -1,
      -1, -1, -1, -1, -1, -1, 1, -1, -1,
      0, -1, -1, -1, -1, -1, 
      -1, 1, -1, -1]

### 2. Light GBM

In [19]:
train_lgb = train.copy()
train_lgb['Gender'] = train_lgb['Gender'].astype('category')
test_lgb = test.copy()
test_lgb['Gender'] = test_lgb['Gender'].astype('category')

ne1, md1, ml1, mf1, ms1, lr1, la1, ll2, ll1, ar1, ar2 = [], [], [], [], [], [], [], [], [], [], []

for ne2 in tqdm([2,5,10], colour='white'):
    for md2 in [2,5,10]:
        for ml2 in [100,200,500]:
            for mf2 in [0.2,0.5,1.0]:      
                for ms2 in [0.2,0.5,1.0]:  
                    for lr2 in [0.2,0.5,1.0]:
                        for la2 in [1.0,5.0,10.0]:  
                            for ll2 in [0.1,0.5,1.0]:

                                ne1.append(ne2); md1.append(md2); ml1.append(ml2); 
                                mf1.append(mf2); ms1.append(ms2); lr1.append(lr2); 
                                la1.append(la2); ll1.append(ll2); 

                                x = train_lgb.drop(['CustomerID', 'Churn'], axis=1)
                                y = train_lgb[['Churn']]
                                train_set = lgb.Dataset(x, label=y, categorical_feature=['Gender'], free_raw_data=False)

                                params = {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'max_depth': md2,
                                            'min_data_in_leaf': ml2, 'feature_fraction': mf2, 'bagging_fraction': ms2,
                                            'bagging_freq': 1, 'learning_rate': lr2, 'lambda_l1': la2, 'lambda_l2': ll2, 
                                            'monotone_constraints': c2, 'monotone_constraints_method': 'advanced', 'verbosity': -1, 'seed': 42}
                                model = lgb.train(params, train_set, num_boost_round=ne2) 
                                p = model.predict(x)
                                ar1.append(roc_auc_score(y, p))

                                x = test_lgb.drop(['CustomerID', 'Churn'], axis=1)
                                y = test_lgb[['Churn']]
                                p = model.predict(x)
                                ar2.append(roc_auc_score(y, p))

gb = pd.DataFrame({'n_estimators': ne1, 'max_depth': md1, 'min_data_in_leaf': ml1, 'feature_fraction': mf1, 
                   'bagging_fraction': ms1, 'learning_rate':lr1, 'lambda_l1': la1, 'lambda_l2': ll1, 
                   'train_auroc': ar1, 'test_auroc': ar2})
gb = gb.sort_values('test_auroc', ascending=False).head(15)
gb

100%|██████████| 3/3 [03:30<00:00, 70.05s/it]


,n_estimators,max_depth,min_data_in_leaf,feature_fraction,bagging_fraction,learning_rate,lambda_l1,lambda_l2,train_auroc,test_auroc
1091,2,5,200,0.5,0.5,0.5,1.0,1.0,0.535687,0.521458
1820,2,10,200,0.5,0.5,0.5,1.0,1.0,0.535687,0.521458
1819,2,10,200,0.5,0.5,0.5,1.0,0.5,0.535687,0.521458
1818,2,10,200,0.5,0.5,0.5,1.0,0.1,0.535687,0.521458
1089,2,5,200,0.5,0.5,0.5,1.0,0.1,0.535687,0.521458
1090,2,5,200,0.5,0.5,0.5,1.0,0.5,0.535687,0.521458
1080,2,5,200,0.5,0.5,0.2,1.0,0.1,0.539622,0.519243
1082,2,5,200,0.5,0.5,0.2,1.0,1.0,0.539622,0.519243
1081,2,5,200,0.5,0.5,0.2,1.0,0.5,0.539622,0.519243
1811,2,10,200,0.5,0.5,0.2,1.0,1.0,0.539622,0.519243


In [20]:
train_lgb = train.copy()
train_lgb['Gender'] = train_lgb['Gender'].astype('category')
test_lgb = test.copy()
test_lgb['Gender'] = test_lgb['Gender'].astype('category')
val_lgb = val.copy()
val_lgb['Gender'] = val_lgb['Gender'].astype('category')

x = train_lgb.drop(['CustomerID', 'Churn'], axis=1)
y = train_lgb[['Churn']]
params = {'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'max_depth': 5,
            'min_data_in_leaf': 200, 'feature_fraction': 0.5, 'bagging_fraction': 0.5,
            'bagging_freq': 1, 'learning_rate': 0.5, 'lambda_l1': 1.0, 'lambda_l2': 1.0, 
            'monotone_constraints': c2, 'monotone_constraints_method': 'advanced', 'verbosity': -1, 'seed': 42}
model = lgb.train(params, train_set, num_boost_round=2) 
p = model.predict(x)
print('train shape',train_lgb.shape,'\t\t train avg chrun rate',np.round(np.mean(train_lgb['Churn'])*100,1),'\t\t train auroc',np.round(roc_auc_score(y, p),3)) 
        
x = test_lgb.drop(['CustomerID', 'Churn'], axis=1)
y = test_lgb[['Churn']]
p = model.predict(x)
print('test shape',test_lgb.shape,'\t\t test avg chrun rate',np.round(np.mean(test_lgb['Churn'])*100,1),'\t\t test auroc',np.round(roc_auc_score(y, p),3)) 

x = val_lgb.drop(['CustomerID', 'Churn'], axis=1)
y = val_lgb[['Churn']]
p = model.predict(x)
print('val shape',val_lgb.shape,'\t\t val avg chrun rate',np.round(np.mean(val_lgb['Churn'])*100,1),'\t\t val auroc',np.round(roc_auc_score(y, p),3)) 

train shape (5120, 30) 		 train avg chrun rate 50.2 		 train auroc 0.536
test shape (2462, 30) 		 test avg chrun rate 49.4 		 test auroc 0.521
val shape (2418, 30) 		 val avg chrun rate 49.0 		 val auroc 0.501
